In [6]:
import pandas as pd
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [7]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'].isnull().sum()

np.int64(11)

In [10]:
df.describe()

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,Churn Value,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7032.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,2283.300441,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,2266.771362,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,18.800000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,401.450000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,1397.475000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,3794.737500,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,8684.800000,1.000000,100.000000,6500.000000


In [11]:
df['Churn Reason'].value_counts()

Churn Reason
Attitude of support person                   192
Competitor offered higher download speeds    189
Competitor offered more data                 162
Don't know                                   154
Competitor made better offer                 140
Attitude of service provider                 135
Competitor had better devices                130
Network reliability                          103
Product dissatisfaction                      102
Price too high                                98
Service dissatisfaction                       89
Lack of self-service on Website               88
Extra data charges                            57
Moved                                         53
Limited range of services                     44
Long distance charges                         44
Lack of affordable download/upload speed      44
Poor expertise of phone support               20
Poor expertise of online support              19
Deceased                                       6
Name: c

In [12]:
df['CLTV'].describe()

count    7043.000000
mean     4400.295755
std      1183.057152
min      2003.000000
25%      3469.000000
50%      4527.000000
75%      5380.500000
max      6500.000000
Name: CLTV, dtype: float64

In [13]:
df['Total Charges'] = df['Total Charges'].fillna(0)

In [14]:
df.groupby('Churn Label')['CLTV'].describe()

,count,mean,std,min,25%,50%,75%,max
Churn Label,,,,,,,,
No,5174.0,4490.921337,1167.703198,2003.0,3643.75,4620.0,5434.75,6500.0
Yes,1869.0,4149.414660,1189.370707,2003.0,3101.00,4238.0,5166.00,6484.0


In [15]:
def reason_category(reason):
    if pd.isnull(reason):
        return None
    competitor = ['Competitor offered higher download speeds','Competitor offered more data',
                  'Competitor made better offer','Competitor had better devices']
    service = ['Attitude of support person','Attitude of service provider','Network reliability',
               'Product dissatisfaction','Service dissatisfaction','Lack of self-service on Website',
               'Poor expertise of phone support','Poor expertise of online support']
    price = ['Price too high','Extra data charges','Long distance charges',
             'Lack of affordable download/upload speed']
    if reason in competitor: return 'Competitor'
    elif reason in service: return 'Service/Support'
    elif reason in price: return 'Price'
    else: return 'Other'

df['Reason Category'] = df['Churn Reason'].apply(reason_category)
df['CLTV Tier'] = pd.qcut(df['CLTV'], q=3, labels=['Low','Medium','High'])

pd.crosstab(df['CLTV Tier'], df['Reason Category'], normalize='index')

Reason Category,Competitor,Other,Price,Service/Support
CLTV Tier,,,,
Low,0.328377,0.140025,0.132590,0.399009
Medium,0.296625,0.142096,0.136767,0.424512
High,0.378758,0.128257,0.118236,0.374749


In [16]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['CLTV Tier'], df['Reason Category'])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square: {chi2:.2f}, p-value: {p:.4f}")

Chi-square: 8.28, p-value: 0.2182


In [17]:
from scipy.stats import ttest_ind
churned = df[df['Churn Label'] == 'Yes']['CLTV']
retained = df[df['Churn Label'] == 'No']['CLTV']
t_stat, p_val = ttest_ind(churned, retained)
print(f"t-stat: {t_stat:.2f}, p-value: {p_val:.4f}")

t-stat: -10.78, p-value: 0.0000


In [18]:
pd.crosstab(df['Contract'], df['Churn Label'], normalize='index')

Churn Label,No,Yes
Contract,,
Month-to-month,0.572903,0.427097
One year,0.887305,0.112695
Two year,0.971681,0.028319


In [19]:
df.groupby('Churn Label')[['Tenure Months', 'Monthly Charges']].mean()

,Tenure Months,Monthly Charges
Churn Label,,
No,37.569965,61.265124
Yes,17.979133,74.441332


In [20]:
df.to_csv('../data/processed/Telco_churn_cleaned.csv', index=False)

In [21]:
import os
print(os.path.exists('../data/processed/telco_churn_cleaned.csv'))

True
